# Multi-Crop Leaf Disease Detection - Google Colab Training

This notebook trains MobileNetV2 and EfficientNet-Lite0 models using your dataset in Google Drive.

**Prerequisites:**
- Runtime -> Change runtime type -> GPU
- Dataset split into train/val/test folders in `/My Drive/leaf_data/processed/`

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive  # pyright: ignore[reportMissingImports]
drive.mount('/content/drive')
print("Drive mounted successfully!")

## Step 2: Verify Dataset Exists

In [ ]:
import os

dataset_base = "/content/drive/MyDrive/leaf_data/processed"

# Check folders exist
for split in ['train', 'val', 'test']:
    split_path = os.path.join(dataset_base, split)
    if os.path.exists(split_path):
        n_classes = len([d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d))])
        print(f"✓ {split}: {n_classes} classes")
    else:
        print(f"✗ {split}: NOT FOUND")

print(f"\nDataset base: {dataset_base}")

## Step 3: Clone Project Repository

In [ ]:
%cd /content

# Clone repo (replace with your GitHub repo URL)
!git clone https://github.com/hit1363/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System.git

%cd /content/multi-crop-leaf-disease-detection
print("Repository cloned successfully!")

## Step 4: Install Dependencies

In [ ]:
%pip install -q -r requirements.txt
print("Dependencies installed successfully!")

## Step 5: Configure Training for MobileNetV2

In [ ]:
import yaml
import os

# Paths (already configured for your dataset structure)
dataset_base = "/content/drive/MyDrive/leaf_data/processed"
output_base = "/content/drive/MyDrive/leaf_outputs"

# Load config
cfg_path = "training/config_mobilenetv2.yaml"
with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

# Set dataset paths
cfg["dataset"]["data_dir"] = dataset_base
cfg["dataset"]["train_dir"] = f"{dataset_base}/train"
cfg["dataset"]["val_dir"] = f"{dataset_base}/val"
cfg["dataset"]["test_dir"] = f"{dataset_base}/test"

# Auto-count classes from training split
num_classes = len([
    d for d in os.listdir(cfg["dataset"]["train_dir"])
    if os.path.isdir(os.path.join(cfg["dataset"]["train_dir"], d))
])
cfg["model"]["num_classes"] = num_classes

# Set output paths to save in Drive
cfg["export"]["save_dir"] = f"{output_base}/models"
cfg["callbacks"]["tensorboard"]["log_dir"] = f"{output_base}/logs"
cfg["callbacks"]["csv_logger"]["filename"] = f"{output_base}/results/training_log_mobilenetv2.csv"

# Save updated config
with open(cfg_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"✓ Classes: {cfg['model']['num_classes']}")
print(f"✓ Train: {cfg['dataset']['train_dir']}")
print(f"✓ Val: {cfg['dataset']['val_dir']}")
print(f"✓ Test: {cfg['dataset']['test_dir']}")
print(f"✓ Models save to: {cfg['export']['save_dir']}")

## Step 6: Train MobileNetV2

In [ ]:
%cd /content/multi-crop-leaf-disease-detection/training

!python train.py --config config_mobilenetv2.yaml

## (Optional) Step 7: Configure & Train EfficientNet-Lite0

In [ ]:
import yaml
import os

# Load EfficientNet config
cfg_path = "config_efficientnet_lite0.yaml"
with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

dataset_base = "/content/drive/MyDrive/leaf_data/processed"
output_base = "/content/drive/MyDrive/leaf_outputs"

# Set dataset paths
cfg["dataset"]["data_dir"] = dataset_base
cfg["dataset"]["train_dir"] = f"{dataset_base}/train"
cfg["dataset"]["val_dir"] = f"{dataset_base}/val"
cfg["dataset"]["test_dir"] = f"{dataset_base}/test"

# Auto-count classes
num_classes = len([
    d for d in os.listdir(cfg["dataset"]["train_dir"])
    if os.path.isdir(os.path.join(cfg["dataset"]["train_dir"], d))
])
cfg["model"]["num_classes"] = num_classes

# Set output paths
cfg["export"]["save_dir"] = f"{output_base}/models"
cfg["callbacks"]["tensorboard"]["log_dir"] = f"{output_base}/logs"
cfg["callbacks"]["csv_logger"]["filename"] = f"{output_base}/results/training_log_efficientnet_lite0.csv"

# Save updated config
with open(cfg_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("✓ EfficientNet-Lite0 config updated")
print(f"✓ Classes: {cfg['model']['num_classes']}")

In [ ]:
!python train.py --config config_efficientnet_lite0.yaml

## Step 8: List Trained Models

In [ ]:
import os

models_dir = "/content/drive/MyDrive/leaf_outputs/models"

for arch in ['mobilenetv2', 'efficientnet_lite0']:
    arch_dir = os.path.join(models_dir, arch)
    if os.path.exists(arch_dir):
        print(f"\n{arch}:")
        for f in os.listdir(arch_dir):
            print(f"  - {f}")
    else:
        print(f"\n{arch}: No models found yet")

## Step 9: Evaluate a Trained Model

In [ ]:
import os
import subprocess

# Find the latest MobileNetV2 model
mobilenetv2_dir = "/content/drive/MyDrive/leaf_outputs/models/mobilenetv2"

if os.path.exists(mobilenetv2_dir):
    models = sorted([f for f in os.listdir(mobilenetv2_dir) if f.endswith('.h5')])
    if models:
        latest_model = models[-1]
        model_path = os.path.join(mobilenetv2_dir, latest_model)
        
        print(f"Evaluating: {latest_model}\n")
        
        training_dir = '/content/multi-crop-leaf-disease-detection/training'
        os.chdir(training_dir)
        subprocess.run(['python', 'evaluate.py', '--model', model_path, '--config', 'config_mobilenetv2.yaml'])
    else:
        print("No .h5 models found yet")
else:
    print(f"Directory not found: {mobilenetv2_dir}")

## Step 10: View Training Logs from Drive

In [ ]:
import pandas as pd
import os

results_dir = "/content/drive/MyDrive/leaf_outputs/results"

if os.path.exists(results_dir):
    csv_files = [f for f in os.listdir(results_dir) if f.endswith('.csv')]
    if csv_files:
        for csv_file in csv_files:
            csv_path = os.path.join(results_dir, csv_file)
            print(f"\n=== {csv_file} ===")
            df = pd.read_csv(csv_path)
            print(df.tail(10))  # Show last 10 rows
    else:
        print("No CSV files found in results directory yet")
else:
    print("Results directory not found yet")